In [2]:
import pandas as pd

xls = pd.ExcelFile("../data/raw/cbi_a18_raw.xls")

print(xls.sheet_names)

['Table A.18', 'Table A.18 - Outstanding', 'Table A.18 - Transactions', 'Table A.18 Growth Rates']


In [3]:
xls = pd.ExcelFile("../data/raw/cbi_a18_raw.xls")

a18_outstanding = pd.read_excel(
    "../data/raw/cbi_a18_raw.xls",
    sheet_name="Table A.18 - Outstanding",
    header=None
)

print(a18_outstanding.shape)
a18_outstanding.head(15)

(102, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Table A.18 Credit Advanced to and Deposits fro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Total Lending,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Total Deposits
3,NaN,NaN,NaN,Lending for House Purchase,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other Personal,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,Floating Rate,NaN,NaN,NaN,Fixed Rate,NaN,NaN,NaN,NaN,Finance for investment,Finance for Other Purposes,NaN
5,NaN,NaN,NaN,NaN,NaN,Standard Variable,Tracker,Up to 1 year fixed,NaN,Over 1 and up to 3 years,Over 3 and up to 5 years,Over 5 years,NaN,NaN,NaN,NaN
6,Series Code,NaN,1243,777,1244,1245,1246,1247,1248,1249,1250,1251,1252,1253,1254,962
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Outstanding amounts - € million,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


For your MVP, I recommend Series Code 1243 — Total Lending, measured in € million outstanding. This gives us a broad household-credit indicator that fits nicely with your mortgage rates, property prices, inflation, and arrears story.


# Step 1 — Extract Series 1243

In [4]:
a18_outstanding = pd.read_excel(
    "../data/raw/cbi_a18_raw.xls",
    sheet_name="Table A.18 - Outstanding",
    header=None
)

household_credit = a18_outstanding.iloc[10:, [1, 2]].copy()

household_credit.columns = ["date", "household_credit"]

household_credit["date"] = pd.to_datetime(
    household_credit["date"],
    errors="coerce"
)

household_credit["household_credit"] = pd.to_numeric(
    household_credit["household_credit"],
    errors="coerce"
)

household_credit = household_credit.dropna(
    subset=["date", "household_credit"]
)

household_credit = household_credit.sort_values(
    "date"
).reset_index(drop=True)

print(household_credit.head())
print(household_credit.tail())
print(household_credit.shape)

        date  household_credit
0    1-12-01            108136
1 2003-03-31             57764
2 2003-06-30             59857
3 2003-09-30             63952
4 2003-12-31             68341
         date  household_credit
87 2024-09-30             97621
88 2024-12-31             98597
89 2025-03-31             99184
90 2025-06-30            104494
91 2025-09-30            106527
(92, 2)


In [7]:
household_credit = household_credit[
    household_credit["date"].notna()
].copy()

# Keep only sensible reporting dates
household_credit = household_credit[
    household_credit["date"] >= "2003-03-31"
].copy()

household_credit = household_credit.sort_values(
    "date"
).reset_index(drop=True)

print(household_credit.head())
print(household_credit.tail())
print(household_credit.shape)

        date  household_credit
0 2003-03-31             57764
1 2003-06-30             59857
2 2003-09-30             63952
3 2003-12-31             68341
4 2004-03-31             72072
         date  household_credit
86 2024-09-30             97621
87 2024-12-31             98597
88 2025-03-31             99184
89 2025-06-30            104494
90 2025-09-30            106527
(91, 2)


# Step 2 — Validate it

In [8]:
print("Missing dates:", household_credit["date"].isna().sum())
print("Missing values:", household_credit["household_credit"].isna().sum())
print("Duplicate dates:", household_credit["date"].duplicated().sum())

print("Date range:")
print(household_credit["date"].min())
print(household_credit["date"].max())

print("Credit range:")
print(household_credit["household_credit"].min())
print(household_credit["household_credit"].max())

Missing dates: 0
Missing values: 0
Duplicate dates: 0
Date range:
2003-03-31 00:00:00
2025-09-30 00:00:00
Credit range:
57764
149670


# save it!

In [9]:
processed_path = "../data/processed/household_credit.csv"

household_credit.to_csv(
    processed_path,
    index=False
)

print("Processed data saved to:", processed_path)

Processed data saved to: ../data/processed/household_credit.csv
